# 3 · Graph of Thought — Combine The Pieces

**The problem:** you have several documents and you need **one** summary.
If you paste them all in at once, the middle ones get ignored.
If you feed them one at a time, the last one dominates.
And every time a model "combines" two texts, it quietly drops the numbers
and smooths over disagreements.

**The fix:** summarise each document **separately**, then join the summaries
**in pairs**, following written rules — keep every point, keep every number,
and when two sources disagree, **say so**.

**The scenario:** four regional managers each sent a short report about the same
product launch. You must write one honest summary for leadership.

Note what is planted in the reports:
- West says customer satisfaction is **8/10**, North reports **45-minute** support waits — those clash.
- South says sales are **up 12%**, East says **down 3%** — those clash directly.

A good summary must keep all four numbers and flag both disagreements.

In [ ]:
# ---- Step 0: check the kernel, then install what is missing ----
# Run this first. It works on Colab, on a fresh laptop,
# and it tells you plainly if the notebook is running the wrong Python.

import importlib.util, subprocess, sys

if sys.version_info < (3, 10):
    print("STOP - this notebook needs Python 3.10 or newer.")
    print("This kernel is Python", sys.version.split()[0], "at", sys.executable)
    print()
    print("Fix it like this:")
    print("  In Jupyter / VS Code : Kernel > Change Kernel, and pick the one from")
    print("                         structured_prompting/.venv")
    print("  On Colab             : Runtime > Restart session, then run this cell again")
    raise SystemExit("Wrong Python version - see the message above.")

REQUIRED = [
    ("openai", "openai==2.53.0"),
    ("dotenv", "python-dotenv==1.2.2"),
]

missing = [pkg for mod, pkg in REQUIRED if importlib.util.find_spec(mod) is None]

if missing:
    print("Installing:", ", ".join(missing))
    print("(a minute the first time, nothing the next time)")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("Done.")
else:
    print("All libraries already here.")

print("Python", sys.version.split()[0], "at", sys.executable)


In [ ]:
# ---- Setup: run this cell first ----
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    try:
        # Google Colab: add OPENAI_API_KEY in the Secrets panel (the key icon, left)
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except Exception:
        import getpass  # last resort: type it here, it is not saved anywhere
        os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI key: ")

client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

def ask(prompt, temperature=0):
    """Send one prompt to the model and return its reply as plain text."""
    reply = client.chat.completions.create(
        model=MODEL,
        temperature=temperature,
        messages=[{"role": "user", "content": prompt}],
    )
    return reply.choices[0].message.content

print("Setup done. Using model:", MODEL)

In [ ]:
# The four regional reports (in real life these would be emails or documents)

reports = {
"West": """The launch went smoothly in the West region.
Customer satisfaction in our follow-up survey is 8 out of 10.
Retail partners reordered within two weeks.
Staff training was completed on schedule.""",

"North": """The North region has had a difficult launch.
Customers are waiting up to 45 minutes for support calls.
Two of our five distributors delayed their first order.
Social media mentions in the region are mostly negative.""",

"South": """Strong results in the South region.
Sales are up 12% compared to the previous product's first month.
The introductory pricing was well received.
No supply problems so far.""",

"East": """The East region is behind expectations.
Sales are down 3% compared to the previous product's first month.
Several retailers say the price point is too high for this market.
Stock is sitting longer on shelves than planned.""",
}

for region, report in reports.items():
    print(f"--- {region} ---")
    print(report)
    print()

---
## Example 1 · The wrong way — "combine these reports"

One call, no rules. Watch what happens to the numbers.

In [ ]:
wrong_prompt = "Combine these four regional reports into one short summary for leadership:\n\n"
for region, report in reports.items():
    wrong_prompt += f"{region} region:\n{report}\n\n"

print(ask(wrong_prompt))

Read the result carefully and check:

- Is the **8/10** still there? The **45 minutes**? The **+12%** and the **−3%**?
- Or has it become *"results were mixed"* and *"some regions reported challenges"*?

Usually most numbers are gone, and the two direct disagreements have been
smoothed into vague middle-ground phrases. Leadership reads it, learns nothing,
and the 45-minute support problem stays hidden.

> **Lesson:** "combine these" is really "shorten these". Every combine loses detail —
> and the most important detail, a disagreement between sources, is always
> the first thing to go.

---
## Example 2 · The right way — separate, then merge with rules

Three steps:

1. Turn each report into a **list of findings** — one line each, numbers kept.
2. Merge the lists **in pairs**, with written rules.
3. The rules force disagreements into the open instead of averaging them away.

### Step 1 — one list of findings per report

In [ ]:
def extract_findings(region, report):
    prompt = f"""Turn this regional report into a list of findings.

Rules:
- One finding per line, starting with a dash.
- Keep every number exactly as written.
- Start every line with the region name in brackets, like [West].
- Do not add opinions. Do not drop anything.

The report from {region}:
{report}"""
    return ask(prompt)

finding_lists = []
for region, report in reports.items():
    findings = extract_findings(region, report)
    finding_lists.append(findings)
    print(f"--- Findings from {region} ---")
    print(findings)
    print()

### Step 2 — merge in pairs, with rules

We merge West+North, then South+East, then the two results.
Every report goes through the same number of merges — equal weight for everyone.

The rules below are the whole trick. Read them once before running.

In [ ]:
def merge(list_a, list_b):
    prompt = f"""Combine these two lists of findings into one list.

Rules — follow them exactly:
- Keep every finding. Never drop one.
- Never change or remove a number.
- Never replace a number with words like "some" or "mixed".
- If two findings say the same thing, keep one line and name both regions.
- If two findings DISAGREE with each other, keep BOTH lines and add a new line
  directly under them, starting with CONFLICT:, saying in one sentence what
  the disagreement is.
- Add nothing that is not in the lists.

List one:
{list_a}

List two:
{list_b}"""
    return ask(prompt)

merged_west_north = merge(finding_lists[0], finding_lists[1])
merged_south_east = merge(finding_lists[2], finding_lists[3])
final_summary     = merge(merged_west_north, merged_south_east)

print("=== FINAL SUMMARY FOR LEADERSHIP ===")
print()
print(final_summary)

### Check the result

- All four numbers should still be there: **8/10, 45 minutes, +12%, −3%**.
- There should be at least one **CONFLICT:** line for satisfaction
  (West vs North) and one for sales (South vs East).

Those CONFLICT lines are the most valuable output in this notebook.
*"Two of your systems disagree about revenue"* is exactly the sentence
leadership needs to read — and it is exactly the sentence that the
wrong approach deleted.

---
## Summary

1. **What it is:** work on each piece separately, then join the results in pairs
   so every piece carries equal weight.
2. **Join lists, not paragraphs.** A list with numbers and sources survives
   three rounds of merging. A paragraph does not.
3. **Write the rules for disagreement:** keep both sides, mark it CONFLICT,
   never average, never soften.
4. **Try it yourself:** add a fifth report that contradicts the South region's
   pricing claim, and check that a new CONFLICT line appears.